In [1]:
import os

In [2]:
PRJ_DIR = os.path.abspath('')

In [3]:
CODE_DIR = os.path.dirname(PRJ_DIR)

In [4]:
event_data_dir = os.path.join(PRJ_DIR, 'event_data')

In [5]:
# Count number of files in event_data directory
num_files = len(os.listdir(event_data_dir))
print(f"Number of files in event_data directory: {num_files}")

Number of files in event_data directory: 2749


In [33]:
IDlist = {
    '2018-2019' : range(1284741, 1285121),
    '2019-2020' : range(1375927, 1376307),
    '2020-2021' : range(1485184, 1485564),
    '2021-2022' : range(1549539, 1549919),
    '2022-2023' : range(1640674, 1641054),# range(1641049, 1641054),
    '2023-2024' : range(1729190, 1729570),
    '2024-2025' : range(1821049, 1821429),
    '2025-2026' : range(1903117, 1903207)
}

In [34]:
possible_match_ids = []
for season, id_range in IDlist.items():
    possible_match_ids.extend(list(id_range))

In [35]:
# Check for unexpected match IDs in event_data directory
for file_name in os.listdir(event_data_dir):
    match_id = int(file_name.split('.')[0])
    if match_id not in possible_match_ids:
        print(f"Unexpected match ID found: {match_id} in file {file_name}")

In [49]:
# Check for missing match IDs
for match_id in possible_match_ids:
    file_name = f"{match_id}.csv"
    if not os.path.exists(os.path.join(event_data_dir, file_name)):
        print(f"Missing match ID: {match_id}, expected file: {file_name}")

Missing match ID: 1729492, expected file: 1729492.csv


# Try To Merge

In [43]:
# Try To Merge 2 cvs files
import pandas as pd

old_csv_path = os.path.join(CODE_DIR, 'test_data', 'data', 'pregame_data.csv')
new_csv_path = os.path.join(CODE_DIR, 'test_data', 'data', 'pregame_data','pregame_data.csv')

old_df = pd.read_csv(old_csv_path)
new_df = pd.read_csv(new_csv_path)
merged_df = pd.concat([old_df, new_df]).drop_duplicates().reset_index(drop=True)
merged_csv_path = os.path.join(CODE_DIR, 'test_data', 'data', 'pregame_data_merged.csv')
merged_df.to_csv(merged_csv_path, index=False)
print(f"Merged CSV saved to {merged_csv_path} with {len(merged_df)} records.")

Merged CSV saved to /home/hungchan/Work/2025.1/Football-in-game-win-probability/code/test_data/data/pregame_data_merged.csv with 3113 records.


In [44]:
merged_df

,match_id,date,home_team,away_team,home_team_id,away_team_id,home_team_elo,away_team_elo
0,1284741,2018-08-12,Arsenal,Man City,13,167,1824.001221,1974.961670
1,1284742,2018-08-11,Bournemouth,Cardiff,183,188,1673.780518,1576.490356
2,1284743,2018-08-11,Fulham,Crystal Palace,170,162,1633.799683,1692.951660
3,1284744,2018-08-11,Huddersfield,Chelsea,166,15,1567.101318,1837.004272
4,1284745,2018-08-12,Liverpool,West Ham,26,29,1917.330200,1673.732788
...,...,...,...,...,...,...,...,...
3108,1903202,2025-10-26,Everton,Tottenham,31,30,1807.931763,1813.797852
3109,1903203,2025-10-24,Leeds,West Ham,19,29,1733.216309,1724.255615
3110,1903204,2025-10-25,Man United,Brighton,32,211,1818.092407,1842.791748
3111,1903205,2025-10-25,Newcastle,Fulham,23,170,1880.464355,1779.720947


In [45]:
# Remove duplicate match IDs in merged_df
merged_df = merged_df.drop_duplicates(subset=['match_id']).reset_index(drop=True)

In [47]:
merged_df.to_csv(merged_csv_path, index=False)
print(f"Merged CSV saved to {merged_csv_path} with {len(merged_df)} records.")

Merged CSV saved to /home/hungchan/Work/2025.1/Football-in-game-win-probability/code/test_data/data/pregame_data_merged.csv with 2748 records.


# Merging 2 files

In [52]:
# Try To Merge 2 cvs files
import pandas as pd

old_csv_path = os.path.join(CODE_DIR, 'data', 'pregame_data.csv')
new_csv_path = os.path.join(CODE_DIR, 'data', 'pregame_data','pregame_data.csv')

old_df = pd.read_csv(old_csv_path)
new_df = pd.read_csv(new_csv_path)
merged_df = pd.concat([old_df, new_df]).drop_duplicates().reset_index(drop=True)
merged_csv_path = os.path.join(CODE_DIR, 'data', 'pregame_data_merged.csv')
merged_df.to_csv(merged_csv_path, index=False)
print(f"Merged CSV saved to {merged_csv_path} with {len(merged_df)} records.")

Merged CSV saved to /home/hungchan/Work/2025.1/Football-in-game-win-probability/code/data/pregame_data_merged.csv with 3114 records.


In [6]:
merged_csv_path = os.path.join(CODE_DIR, 'data', 'pregame_data_merged.csv')


In [13]:
import pandas as pd
merged_df = pd.read_csv(merged_csv_path)

In [14]:
import numpy as np

# Treat empty/whitespace as missing
merged_df['home_team_elo'] = merged_df['home_team_elo'].replace(r'^\s*$', np.nan, regex=True)

# Prefer rows with ELO present, then drop duplicates by match_id
dedup_df = (
    merged_df
    .assign(_elo_present=merged_df['home_team_elo'].notna().astype(int))
    .sort_values(['match_id', '_elo_present'], ascending=[True, False])
    .drop_duplicates(subset=['match_id'], keep='first')
    .drop(columns=['_elo_present'])
    .reset_index(drop=True)
)

# Save and replace the in-memory DataFrame
# dedup_df.to_csv(merged_csv_path, index=False)
# merged_df = dedup_df
# print(f"Deduplicated CSV saved to {merged_csv_path} with {len(merged_df)} records.")

In [57]:
dedup_df.to_csv(merged_csv_path, index=False)

# Get the matchid where elo is missing

In [15]:
# Get the matchid where elo is missing
missing_elo_match_ids = dedup_df[dedup_df['home_team_elo'].isna()]['match_id'].tolist()
print(f"Match IDs with missing ELO: {missing_elo_match_ids}")

Match IDs with missing ELO: []


In [16]:
print(missing_elo_match_ids)

[]


In [18]:
# Sort dedup_df by match_id
dedup_df = dedup_df.sort_values(by='match_id').reset_index(drop=True)

In [20]:
final_csv_path = os.path.join(CODE_DIR, 'data', 'pregame_data_final.csv')


In [ ]:
dedup_df.to_csv(final_csv_path, index=False)

,match_id,date,home_team,away_team,home_team_id,away_team_id,home_team_elo,away_team_elo
0,1284741,2018-08-12,Arsenal,Man City,13,167,1824.001221,1974.961670
1,1284742,2018-08-11,Bournemouth,Cardiff,183,188,1673.780518,1576.490356
2,1284743,2018-08-11,Fulham,Crystal Palace,170,162,1633.799683,1692.951660
3,1284744,2018-08-11,Huddersfield,Chelsea,166,15,1567.101318,1837.004272
4,1284745,2018-08-12,Liverpool,West Ham,26,29,1917.330200,1673.732788
...,...,...,...,...,...,...,...,...
2744,1903202,2025-10-26,Everton,Tottenham,31,30,1807.931763,1813.797852
2745,1903203,2025-10-24,Leeds,West Ham,19,29,1733.216309,1724.255615
2746,1903204,2025-10-25,Man United,Brighton,32,211,1818.092407,1842.791748
2747,1903205,2025-10-25,Newcastle,Fulham,23,170,1880.464355,1779.720947
